In [ ]:
import json
import joblib
from pathlib import Path

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------
# Reproduce the saved 15 bp CatBoost full-retraining run
# ---------------------------------------------------------------------
WORK_DIR = Path.cwd()
if not (WORK_DIR / "lightning_logs").is_dir():
    WORK_DIR = WORK_DIR / "8. Early-Warning Model"
EXPERIMENT_NAME = "cv_model_comparison_2026-09-23_15bp_full_retraining"
experiment_dir = WORK_DIR / "lightning_logs" / EXPERIMENT_NAME
run_dirs = sorted(experiment_dir.glob("catboost_threshold_15_alpha_0.3_fullfeatures_*"))
if len(run_dirs) != 1:
    raise FileNotFoundError(f"Expected one CatBoost run in {experiment_dir}; found {run_dirs}")
RUN_DIR = run_dirs[0]
with (RUN_DIR / "hparams.json").open(encoding="utf-8") as f:
    hparams = json.load(f)
if hparams["model_name"] != "catboost" or float(hparams["target_threshold"]) != 15:
    raise ValueError("The selected run is not the expected 15 bp CatBoost model.")
DATA_PATH = Path(hparams["dataset_path"])
if not DATA_PATH.is_absolute():
    DATA_PATH = WORK_DIR / DATA_PATH
TARGET_COL = "target"
TIME_COL = "timestamp"

TEST_FRAC = float(hparams["test_size"])

df = pd.read_parquet(DATA_PATH)

# Match run_full_training.py exactly: timestamp comes from the index.
df[TIME_COL] = df.index

# Recreate the seven lag features before the chronological test split.
for k in range(1, 8):
    df[f"depeg_bps_lag{k}h"] = df["depeg_bps"].shift(k)

df = df.dropna().copy()

df[TIME_COL] = pd.to_datetime(df[TIME_COL])
df = df.sort_values(TIME_COL).reset_index(drop=True)

if TARGET_COL not in df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found in dataset.")

FEATURES = [c for c in df.columns if c not in [TIME_COL, TARGET_COL]]

X = df[FEATURES].copy()
y = df[TARGET_COL].astype(int).copy()
non_numeric_cols = X.select_dtypes(exclude=[np.number, "bool"]).columns.tolist()
if non_numeric_cols:
    raise ValueError(
        "The saved CatBoost model requires numeric features. "
        f"Non-numeric columns found: {non_numeric_cols}"
    )

print(f"Run: {RUN_DIR}")
print(f"Loaded dataset: {DATA_PATH}")
print(f"Rows after lag/dropna: {len(df):,}")
print(f"Number of features: {len(FEATURES):,}")
print("Target distribution:")
print(y.value_counts().sort_index())

In [ ]:
import matplotlib.pyplot as plt
import shap
from catboost import CatBoostClassifier, Pool

# ---------------------------------------------------------------------
# Load the exact model, feature order, scaler, and operating threshold
# ---------------------------------------------------------------------
MODEL_PATH = RUN_DIR / "artifacts/models/model.joblib"
SCALER_PATH = RUN_DIR / "artifacts/models/preprocess_scaler.joblib"
SIGNATURE_PATH = RUN_DIR / "artifacts/models/signature.json"
THRESHOLD_PATH = RUN_DIR / "artifacts/reports/operating_threshold.json"
PREDICTIONS_PATH = RUN_DIR / "artifacts/predictions/test_pred_proba.parquet"
OUTPUT_DIR = RUN_DIR / "artifacts/shap_details"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for artifact in (MODEL_PATH, SIGNATURE_PATH, THRESHOLD_PATH, PREDICTIONS_PATH):
    if not artifact.exists():
        raise FileNotFoundError(f"Full-retraining artifact not found: {artifact}")

with SIGNATURE_PATH.open(encoding="utf-8") as f:
    signature = json.load(f)
with THRESHOLD_PATH.open(encoding="utf-8") as f:
    threshold_report = json.load(f)

model_features = signature["input_columns"]
if model_features != FEATURES:
    raise ValueError("Feature order differs from the full-retraining model signature.")
missing_features = [c for c in model_features if c not in X.columns]
if missing_features:
    raise ValueError(
        "Current dataset is missing features required by the saved model: "
        f"{missing_features[:10]}{'...' if len(missing_features) > 10 else ''}"
    )

test_size = int(TEST_FRAC * len(df))
X_test_raw = X.iloc[len(df) - test_size:][model_features].copy()
test_timestamps = df[TIME_COL].iloc[len(df) - test_size:].reset_index(drop=True)
model = joblib.load(MODEL_PATH)
if not isinstance(model, CatBoostClassifier):
    raise TypeError(f"Expected CatBoostClassifier, found {type(model).__name__}")
print(f"Loaded model from: {MODEL_PATH}")
print(f"Model type: {type(model).__name__}")
print(f"Feature count used by model: {len(model_features)}")

# ---------------------------------------------------------------------
# Apply the training scaler to the same test rows as run_full_training.py
# ---------------------------------------------------------------------
if SCALER_PATH.exists():
    scaler = joblib.load(SCALER_PATH)
    X_test = pd.DataFrame(
        scaler.transform(X_test_raw),
        columns=model_features,
        index=X_test_raw.index,
    )
    print(f"Applied scaler from: {SCALER_PATH}")
else:
    scaler = None
    X_test = X_test_raw
    print("No scaler artifact found. Using raw feature matrix.")

saved_predictions = pd.read_parquet(PREDICTIONS_PATH)
if len(saved_predictions) != len(X_test):
    raise ValueError("Saved prediction count does not match the recreated test split.")
if not np.array_equal(
    pd.to_datetime(saved_predictions["timestamp"], utc=True).to_numpy(),
    pd.to_datetime(test_timestamps, utc=True).to_numpy(),
):
    raise ValueError("Saved prediction timestamps do not match the recreated test split.")
y_pred_proba = model.predict_proba(X_test)[:, 1]
np.testing.assert_allclose(y_pred_proba, saved_predictions["proba_depeg"], rtol=1e-6, atol=1e-8)
thresh = float(threshold_report["operating_threshold"])
np.testing.assert_allclose(
    (y_pred_proba >= thresh).astype(int),
    saved_predictions["y_pred_at_operating_threshold"],
)
warning_mask = y_pred_proba >= thresh
print(f"Test rows: {len(X_test):,}; operating threshold: {thresh:.6f}")
print(f"Warnings above threshold: {warning_mask.sum():,}; outputs: {OUTPUT_DIR}")

In [ ]:
# Binary CatBoost returns one SHAP value per feature, in raw log-odds units.
explainer = shap.TreeExplainer(model, model_output="raw")
sv = explainer(X_test)
if sv.values.shape != X_test.shape:
    raise ValueError(f"Expected SHAP shape {X_test.shape}, got {sv.values.shape}")
sv = shap.Explanation(
    values=sv.values, base_values=sv.base_values, data=X_test.to_numpy(),
    feature_names=model_features,
)
raw_margin = model.predict(X_test, prediction_type="RawFormulaVal")
np.testing.assert_allclose(sv.values.sum(axis=1) + sv.base_values, raw_margin, rtol=1e-4, atol=1e-4)
SHAP_VALUES_PATH = OUTPUT_DIR / "shap_values_test.joblib"
joblib.dump(sv, SHAP_VALUES_PATH)
print(f"Saved test-set SHAP values to {SHAP_VALUES_PATH}")

In [ ]:
# Use the saved test-row interaction matrix when it matches this run.
# Compute it in batches only if no complete matrix has been saved yet.
INTERACTION_BATCH_SIZE = 64
INTERACTION_PATH = OUTPUT_DIR / "shap_interaction_values_test.npy"
INTERACTION_META_PATH = OUTPUT_DIR / "shap_interaction_values_test.json"
n_rows, n_features = X_test.shape
interaction_metadata = {
    "run_dir": str(RUN_DIR.resolve()),
    "shape": [n_rows, n_features, n_features],
    "feature_names": model_features,
    "units": "raw_log_odds",
    "first_test_timestamp": str(test_timestamps.iloc[0]),
    "last_test_timestamp": str(test_timestamps.iloc[-1]),
}
if INTERACTION_PATH.exists() and INTERACTION_META_PATH.exists():
    with INTERACTION_META_PATH.open(encoding="utf-8") as f:
        saved_metadata = json.load(f)
    if saved_metadata != interaction_metadata:
        raise ValueError("Saved SHAP interactions belong to a different run or test split.")
    print(f"Loading saved interactions from {INTERACTION_PATH}")
else:
    partial_path = OUTPUT_DIR / "shap_interaction_values_test.partial.npy"
    interaction_file = np.lib.format.open_memmap(
        partial_path, mode="w+", dtype=np.float64,
        shape=(n_rows, n_features, n_features),
    )
    for start in range(0, n_rows, INTERACTION_BATCH_SIZE):
        stop = min(start + INTERACTION_BATCH_SIZE, n_rows)
        batch = X_test.iloc[start:stop]
        raw_interactions = explainer.shap_interaction_values(Pool(batch))
        if isinstance(raw_interactions, list):
            raw_interactions = raw_interactions[1] if len(raw_interactions) == 2 else raw_interactions[0]
        interactions = np.asarray(raw_interactions)
        if interactions.ndim == 4 and interactions.shape[-1] == 2:
            interactions = interactions[..., 1]
        elif interactions.ndim == 4 and interactions.shape[1] == 2:
            interactions = interactions[:, 1, :, :]
        if interactions.shape == (len(batch), n_features + 1, n_features + 1):
            interactions = interactions[:, :-1, :-1]
        if interactions.shape != (len(batch), n_features, n_features):
            raise ValueError(f"Unexpected SHAP interaction shape: {interactions.shape}")
        np.testing.assert_allclose(interactions.sum(axis=2), sv.values[start:stop], rtol=1e-4, atol=1e-4)
        interaction_file[start:stop] = interactions
        interaction_file.flush()
        print(f"Saved interactions for test rows {start:,}--{stop:,} of {n_rows:,}", end="\r")
    del interaction_file
    partial_path.replace(INTERACTION_PATH)
    with INTERACTION_META_PATH.open("w", encoding="utf-8") as f:
        json.dump(interaction_metadata, f, indent=2)
svint = np.load(INTERACTION_PATH, mmap_mode="r")
if svint.shape != tuple(interaction_metadata["shape"]):
    raise ValueError(f"Saved interaction matrix has the wrong shape: {svint.shape}")
for start in range(0, n_rows, INTERACTION_BATCH_SIZE):
    stop = min(start + INTERACTION_BATCH_SIZE, n_rows)
    np.testing.assert_allclose(svint[start:stop].sum(axis=2), sv.values[start:stop], rtol=1e-4, atol=1e-4)
print(f"SHAP interactions ready: {svint.shape}")

In [ ]:
# The diagonal contains main effects. Ordinary SHAP values also include
# shares of pairwise interactions, so use this separate Explanation for plots.
main_effect_values = np.diagonal(svint, axis1=1, axis2=2).copy()
sv_main = shap.Explanation(
    values=main_effect_values, data=X_test.to_numpy(),
    feature_names=model_features,
)
interaction_remainder = sv.values.sum(axis=1) - main_effect_values.sum(axis=1)
np.testing.assert_allclose(
    sv.base_values + main_effect_values.sum(axis=1) + interaction_remainder,
    raw_margin, rtol=1e-4, atol=1e-4,
)
MAIN_EFFECTS_PATH = OUTPUT_DIR / "shap_main_effects_test.joblib"
joblib.dump(sv_main, MAIN_EFFECTS_PATH)
print(f"Saved SHAP main effects to {MAIN_EFFECTS_PATH}")

def plot_main_effect_waterfall(row_idx, filename, top_n=13):
    """Plot leading main effects and show all interactions as a separate term."""
    if not 0 <= row_idx < len(X_test):
        raise IndexError(f"Test row {row_idx} is outside the saved test split.")
    row_effects = main_effect_values[row_idx]
    ranked = np.argsort(np.abs(row_effects))[::-1]
    top = ranked[:top_n]
    other_main = row_effects[ranked[top_n:]].sum()
    values = np.r_[row_effects[top], other_main, interaction_remainder[row_idx]]
    names = [model_features[j] for j in top] + ["Other main effects", "All pairwise interactions"]
    base = float(np.asarray(sv[row_idx].base_values).reshape(-1)[0])
    np.testing.assert_allclose(base + values.sum(), raw_margin[row_idx], rtol=1e-4, atol=1e-4)
    explanation = shap.Explanation(values=values, base_values=base, feature_names=names)
    plt.figure(figsize=(8, 6))
    shap.plots.waterfall(explanation, max_display=len(names), show=False)
    plt.title("SHAP main effects with pairwise interactions (log odds)")
    plt.savefig(OUTPUT_DIR / filename, dpi=300, bbox_inches="tight", transparent=True)
    plt.close()

print(f"Main-effect matrix: {main_effect_values.shape}; figures: {OUTPUT_DIR}")

In [ ]:
shap.plots.beeswarm(sv_main, max_display=20, group_remaining_features=False, order=np.argsort(sv_main.values.std(0))[::-1], plot_size=(11,8), show=False)
plt.title("SHAP main effects on test observations (log odds)")
plt.savefig(OUTPUT_DIR / "shap_beeswarm.png", dpi=300, bbox_inches="tight", transparent=True)
plt.close()

In [ ]:
if not warning_mask.any():
    raise ValueError("No test rows exceed the saved operating threshold.")
positive_order = np.argsort(sv_main.values[warning_mask].mean(axis=0))[::-1]
shap.plots.beeswarm(sv_main, max_display=20, group_remaining_features=False, order=positive_order, plot_size=(12,8), show=False)
plt.title("SHAP main effects ordered by contribution on warnings")
plt.savefig(OUTPUT_DIR / "shap_beeswarm_net_positive.png", dpi=300, bbox_inches="tight", transparent=True)
plt.close()

In [ ]:
plot_main_effect_waterfall(9074, "shap_waterfall_9074.png")

In [ ]:
plot_main_effect_waterfall(9215, "shap_waterfall_9215.png")

In [ ]:
plot_main_effect_waterfall(6412, "shap_waterfall_6412.png")

In [ ]:
feats = ['curve_entropy', 'swap_count_100', 'eth_ATR_24', 'gauge_share_3crv']
fig, axs = plt.subplots(1, len(feats), figsize = (25, 5))
for i, ax in enumerate(axs):
    shap.plots.scatter(sv_main[:, feats[i]], color=sv_main, ax=ax, show=False)
    ax.set_ylabel("SHAP main effect (log odds)")
fig.subplots_adjust(left=0.34, right=0.98, top=0.96, bottom=0.05, hspace=0.45)
fig.tight_layout()
plt.savefig(OUTPUT_DIR / 'shap_scatter_plots.png', dpi=300, bbox_inches="tight", transparent=True)
plt.close(fig)

In [ ]:
feats = ["Gegenbauer_0.3_deg1_vol24", 'Gegenbauer_0.3_deg3_vol24', 'tangent_up', 'tick_width_24h_rolling_median']
fig, axs = plt.subplots(1, len(feats), figsize = (25, 5))
for i, ax in enumerate(axs):
    shap.plots.scatter(sv_main[:, feats[i]], color=sv_main, ax=ax, show=False)
    ax.set_ylabel("SHAP main effect (log odds)")
fig.subplots_adjust(left=0.34, right=0.98, top=0.96, bottom=0.05, hspace=0.45)
fig.tight_layout()
plt.savefig(OUTPUT_DIR / 'shap_scatter_plots_volatility.png', dpi=300, bbox_inches="tight", transparent=True)
plt.close(fig)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import OrderedDict


# ---------------------------------------------------------------------
# 1. Extract SHAP matrix robustly
# ---------------------------------------------------------------------

def get_shap_matrix(sv, positive_class=1):
    """
    Return SHAP main effects as an array of shape (n_samples, n_features).

    Handles:
    - shap.Explanation with values shape (n, p)
    - shap.Explanation with values shape (n, p, K)
    - raw numpy arrays with same shapes
    """
    values = sv.values if hasattr(sv, "values") else sv
    values = np.asarray(values)

    if values.ndim == 2:
        return values

    if values.ndim == 3:
        # Common case for classifiers: (n_samples, n_features, n_classes)
        return values[:, :, positive_class]

    raise ValueError(f"Unexpected SHAP values shape: {values.shape}")


features = list(sv_main.feature_names)
shap_mat = get_shap_matrix(sv_main, positive_class=1)

assert shap_mat.shape[1] == len(features), (
    shap_mat.shape, len(features)
)

shap_df = pd.DataFrame(shap_mat, columns=features)


# ---------------------------------------------------------------------
# 2. Define feature groups
# ---------------------------------------------------------------------

def has_any_prefix(x, prefixes):
    return any(x.startswith(p) for p in prefixes)

def has_any_substring(x, substrings):
    return any(s in x for s in substrings)


feature_groups = OrderedDict()

# Main Uniswap V3 USDC/USDT liquidity-curve shape.
# Includes Gegenbauer coefficients, energy features derived from the
# decomposition, ratios, tangents of swap-size impact curve, and swap-size
# imbalance.
feature_groups["Uniswap V3 liquidity-curve shape"] = [
    f for f in features
    if (
        f.startswith("Gegenbauer_")
        or f.startswith("E_")
        or ("ratio" in f)
        or f in [
            "tangent_up",
            "tangent_down",
            "swap_size_imbalance",
            "tvlUSD_100",
        "tvlUSD_500",
        ]
    )
]

# Curve 3pool broad stablecoin liquidity conditions.
feature_groups["Curve 3pool liquidity conditions"] = [
    f for f in [
        "w_USDC",
        "w_USDT",
        "curve_entropy",
        "gauge_share_3crv",
        "totalValueLockedUSD",
    ]
    if f in features
]

# Broader market conditions: ETH/BTC technical indicators, dollar index,
# FX volatility, sentiment.
feature_groups["Broader market conditions"] = [
    f for f in features
    if (
        f.startswith("eth_")
        or f.startswith("btc_")
        or f in [
            "usd_index",
            "fx_volatility",
            "fear_greed_index",
        ]
    )
]

# Historical stablecoin peg deviation.
feature_groups["Historical peg deviation"] = [
    f for f in features
    if f == "depeg_bps" or f.startswith("depeg_bps_lag")
]

# Liquidity ownership, concentration, and position structure.
# hhi_24h_rolling_mean: concentration of liquidity ownership
# tick_width_24h_rolling_median: typical width of in-range NFPM positions
# n_in_range_log_return: change in number of in-range positions
# weighted_mean_age_hours: age structure of liquidity positions
# tvlUSD_*: stock of liquidity in Uniswap fee-tier pools
feature_groups["Liquidity ownership and position structure"] = [
    f for f in [
        "hhi_24h_rolling_mean",
        "tick_width_24h_rolling_median",
        "n_in_range_log_return",
        "weighted_mean_age_hours",
        
    ]
    if f in features
]

# Market velocity and flows on Uniswap / aggregate DEX activity.
feature_groups["Trading velocity and flows"] = [
    f for f in [
        "swap_count_100",
        "swap_count_500",
        "net_amountUSD_100",
        "net_amountUSD_500",
        "net_amount0",
        "hourlyVolumeUSD",
    ]
    if f in features
]

# AAVE lending market variables.
feature_groups["AAVE lending market conditions"] = [
    f for f in [
        "supplied_USD_usdt",
        "utilisation_rate_usdt",
        "supplied_USD_usdc",
        "utilisation_rate_usdc",
        "liquidation_USD",
    ]
    if f in features
]


# ---------------------------------------------------------------------
# 3. Clean groups: remove duplicates and collect unassigned features
# ---------------------------------------------------------------------

def clean_feature_groups(feature_groups, features, add_other=True):
    """
    Remove duplicate assignments while preserving dictionary order.
    Add an 'Other' group for unassigned model features.
    """
    features = list(features)
    assigned = set()
    clean_groups = OrderedDict()

    for group_name, cols in feature_groups.items():
        cols_clean = []
        for c in cols:
            if c in features and c not in assigned:
                cols_clean.append(c)
                assigned.add(c)

        if len(cols_clean) > 0:
            clean_groups[group_name] = cols_clean

    unassigned = [f for f in features if f not in assigned]

    if add_other and len(unassigned) > 0:
        clean_groups["Other"] = unassigned

    return clean_groups, unassigned


feature_groups, unassigned = clean_feature_groups(feature_groups, features)

print("Feature groups:")
for group, cols in feature_groups.items():
    print(f"\n{group} ({len(cols)} features)")
    print(cols)

print("\nUnassigned features before adding 'Other':")
print(unassigned)


# ---------------------------------------------------------------------
# 4. Compute grouped SHAP main-effect importances
# ---------------------------------------------------------------------

def compute_grouped_shap_importance(
    shap_df,
    feature_groups,
    aggregation="sum_abs",
    mask=None,
    normalize=True,
):
    """
    Compute grouped SHAP main-effect importance.

    Parameters
    ----------
    shap_df : pd.DataFrame
        SHAP main effects, shape (n_samples, n_features).

    feature_groups : dict
        Dictionary mapping group name to list of feature names.

    aggregation : {"sum_abs", "abs_sum", "signed_mean", "positive_mean"}
        - "sum_abs":
            sum_j mean_i |phi_ij| within group.
            Measures total attribution mass of the group.
        - "abs_sum":
            mean_i |sum_j phi_ij| within group.
            Measures net group contribution after cancellation.
        - "signed_mean":
            mean_i sum_j phi_ij.
            Measures average signed contribution.
        - "positive_mean":
            mean_i max(sum_j phi_ij, 0).
            Measures average positive contribution to warning class.

    mask : array-like of bool, optional
        If provided, compute importance only on selected rows,
        e.g. samples where the model triggered an alert.

    normalize : bool
        If True, return percentages summing to 100 for nonnegative
        aggregations.

    Returns
    -------
    importance : pd.Series
        Grouped importance values.
    grouped_contrib : pd.DataFrame
        Group-level main effects, one column per group.
    """
    if mask is not None:
        shap_use = shap_df.loc[mask].copy()
    else:
        shap_use = shap_df.copy()

    grouped_contrib = pd.DataFrame(index=shap_use.index)

    for group, cols in feature_groups.items():
        cols = [c for c in cols if c in shap_use.columns]
        if len(cols) == 0:
            continue
        grouped_contrib[group] = shap_use[cols].sum(axis=1)

    if aggregation == "sum_abs":
        vals = {}
        for group, cols in feature_groups.items():
            cols = [c for c in cols if c in shap_use.columns]
            if len(cols) == 0:
                continue
            vals[group] = shap_use[cols].abs().mean(axis=0).sum()
        importance = pd.Series(vals)

    elif aggregation == "abs_sum":
        importance = grouped_contrib.abs().mean(axis=0)

    elif aggregation == "signed_mean":
        importance = grouped_contrib.mean(axis=0)

    elif aggregation == "positive_mean":
        importance = grouped_contrib.clip(lower=0).mean(axis=0)

    else:
        raise ValueError(
            "aggregation must be one of "
            "{'sum_abs', 'abs_sum', 'signed_mean', 'positive_mean'}"
        )

    importance = importance.sort_values(ascending=True)

    if normalize and aggregation in ["sum_abs", "abs_sum", "positive_mean"]:
        total = importance.sum()
        if total > 0:
            importance = 100 * importance / total

    return importance, grouped_contrib


# ---------------------------------------------------------------------
# 5. Plot helper
# ---------------------------------------------------------------------

def plot_grouped_shap_bar(
    importance,
    title=None,
    xlabel=None,
    figsize=(8, 4.8),
    color="#3B82F6",
    save_path=None,
):
    fig, ax = plt.subplots(figsize=figsize)

    importance.plot(kind="barh", ax=ax, color=color)

    ax.set_title(title or "Grouped SHAP main-effect importance")
    ax.set_xlabel(xlabel or "Share of grouped SHAP main effects (%)")
    ax.set_ylabel("")

    # Add value labels
    xlim = ax.get_xlim()
    width = xlim[1] - xlim[0]
    for i, v in enumerate(importance.values):
        ax.text(
            v + 0.01 * width,
            i,
            f"{v:.1f}",
            va="center",
            fontsize=9,
        )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="x", alpha=0.25)

    plt.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, dpi=300, bbox_inches="tight", transparent = True)

    return fig, ax


# ---------------------------------------------------------------------
# 6. Main grouped plot: total attribution mass by economic group
# ---------------------------------------------------------------------

importance_sum_abs, grouped_contrib = compute_grouped_shap_importance(
    shap_df=shap_df,
    feature_groups=feature_groups,
    aggregation="sum_abs",
    mask=None,
    normalize=True,
)

fig, ax = plot_grouped_shap_bar(
    importance_sum_abs,
    title="Grouped SHAP main effects by economic block",
    xlabel="Share of total mean absolute main effect (%)",
    color="#2563EB",
    save_path=OUTPUT_DIR / "shap_grouped_importance_sum_abs.png",
)

plt.show()


# ---------------------------------------------------------------------
# 7. Alternative plot: net group contribution after within-group cancellation
# ---------------------------------------------------------------------

importance_abs_sum, grouped_contrib = compute_grouped_shap_importance(
    shap_df=shap_df,
    feature_groups=feature_groups,
    aggregation="abs_sum",
    mask=None,
    normalize=True,
)

fig, ax = plot_grouped_shap_bar(
    importance_abs_sum,
    title="Grouped SHAP main effects: net group contributions",
    xlabel="Share of mean absolute net group main effect (%)",
    color="#7C3AED",
    save_path=OUTPUT_DIR / "shap_grouped_importance_abs_sum.png",
)

plt.show()


# ---------------------------------------------------------------------
# 8. Positive contribution to warnings at the saved operating threshold
# ---------------------------------------------------------------------
importance_pos_alerts, grouped_contrib_alerts = compute_grouped_shap_importance(
    shap_df=shap_df,
    feature_groups=feature_groups,
    aggregation="positive_mean",
    mask=warning_mask,
    normalize=True,
)

fig, ax = plot_grouped_shap_bar(
    importance_pos_alerts,
    title="Grouped positive SHAP main effects on model warnings",
    xlabel="Share of positive main effects on alerts (%)",
    color="#DC2626",
    save_path=OUTPUT_DIR / "shap_grouped_positive_alerts.png",
)

plt.show()